# Interactive FIR filter designer and sound processing

Choose the FIR coefficients or a preset, and explore their **impulse response, magnitude, phase, and group delay**. The second interactive section applies **the same designed filter** to white noise or a square wave, so you can see and hear its effect.

The frequency response is computed using `scipy.signal.freqz`, and the sound is processed with `scipy.signal.lfilter`.

*Requires NumPy, SciPy, Matplotlib, ipywidgets, and IPython. Run the cells in order (in JupyterLab, Jupyter Notebook, or VS Code).*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, HTML, display
import ipywidgets as widgets

%matplotlib inline

## 1 · Design and inspect an FIR filter

Choose a preset or enter your own FIR coefficients. The frequency axis runs from **0** to **π rad/sample** (Nyquist). The reference-frequency marker is a measurement cursor, **not** the filter's cutoff frequency. For windowed-sinc filters, the design-cutoff control sets the cutoff.

In [2]:
def parse_coefficients(text):
    # Read a comma- or whitespace-separated list of real FIR coefficients.
    clean = text.replace(";", ",").replace("\n", ",")
    tokens = [p for piece in clean.split(",") for p in piece.split()]
    if not tokens:
        raise ValueError("Enter at least one coefficient, e.g. 0.25, 0.5, 0.25.")
    if len(tokens) > 128:
        raise ValueError("Use at most 128 coefficients for responsive plotting.")
    try:
        b = np.asarray([float(token) for token in tokens], dtype=float)
    except ValueError as exc:
        raise ValueError("Coefficients must be real numbers separated by commas or spaces.") from exc
    if not np.all(np.isfinite(b)):
        raise ValueError("Every coefficient must be finite.")
    if np.all(b == 0):
        raise ValueError("At least one coefficient must be nonzero (zero filter has undefined phase).")
    return b


def make_fir(family, n_taps=11, mix=0.5, symmetry=0.25,
             delay=3, cutoff=0.3, custom="0.25, 0.5, 0.25"):
    # Return (coefficients, description/note) for the selected FIR family.
    if family == "Original: two-tap average":
        return np.array([0.5, 0.5]), "Original example: y[n] = 0.5 x[n] + 0.5 x[n-1]."
    if family == "Adjustable two-tap average":
        return np.array([mix, 1.0 - mix]), "Change b[0] while keeping b[0] + b[1] = 1."
    if family == "N-point moving average":
        return np.ones(n_taps) / n_taps, f"Moving average: b[k] = 1/{n_taps}, for k = 0, ..., {n_taps-1}."
    if family == "First difference":
        return np.array([1.0, -1.0]), "First difference: y[n] = x[n] - x[n-1] (a high-pass shape)."
    if family == "Pure delay":
        return np.r_[np.zeros(delay), 1.0], f"Pure delay: H(e^{{jω}}) = e^{{-jω·{delay}}}; magnitude = 1."
    if family == "Symmetric 3-tap smoother":
        return np.array([symmetry, 1 - 2 * symmetry, symmetry]), "Symmetric b = [a, 1−2a, a], with unit DC gain."
    if family in ("Windowed-sinc low-pass", "Windowed-sinc high-pass"):
        # High-pass designs with nonzero Nyquist gain must have an odd number of taps.
        n_actual = n_taps
        note = "Hamming-window FIR, designed with scipy.signal.firwin."
        if family == "Windowed-sinc high-pass" and n_actual % 2 == 0:
            n_actual += 1
            note += f" High-pass requires odd length here: using {n_actual} taps."
        pass_zero = family == "Windowed-sinc low-pass"
        b = signal.firwin(n_actual, cutoff, pass_zero=pass_zero, window="hamming")
        return b, note + " Cutoff expressed as a fraction of Nyquist (π rad/sample)."
    if family == "Custom coefficients":
        return parse_coefficients(custom), "Custom FIR coefficients entered by the user."
    raise ValueError(f"Unknown family: {family}")


def calculate_response(b, size=2048):
    # Complex response and safe magnitude/phase values used by the plots.
    w, h = signal.freqz(b, worN=size)
    magnitude = np.abs(h)
    db = 20 * np.log10(np.maximum(magnitude, 1e-5))
    phase = np.unwrap(np.angle(h))
    return w, h, magnitude, db, phase

In [3]:
filter_choice = widgets.Dropdown(
    options=[
        "Original: two-tap average", "Adjustable two-tap average",
        "N-point moving average", "First difference", "Pure delay",
        "Symmetric 3-tap smoother", "Windowed-sinc low-pass",
        "Windowed-sinc high-pass", "Custom coefficients"
    ],
    value="Original: two-tap average", description="Filter:",
    style={"description_width": "initial"}, layout=widgets.Layout(width="370px")
)
n_taps_control = widgets.IntSlider(value=11, min=2, max=41, step=1,
    description="Number of taps N:", continuous_update=False,
    style={"description_width": "initial"}, layout=widgets.Layout(width="370px"))
mix_control = widgets.FloatSlider(value=0.5, min=0, max=1, step=0.025,
    description="b[0] = a:", readout_format=".3f", continuous_update=False,
    style={"description_width": "initial"}, layout=widgets.Layout(width="370px"))
symmetry_control = widgets.FloatSlider(value=0.25, min=0, max=0.5, step=0.025,
    description="Outer taps a:", readout_format=".3f", continuous_update=False,
    style={"description_width": "initial"}, layout=widgets.Layout(width="370px"))
delay_control = widgets.IntSlider(value=3, min=0, max=20, step=1,
    description="Delay (samples):", continuous_update=False,
    style={"description_width": "initial"}, layout=widgets.Layout(width="370px"))
cutoff_control = widgets.FloatSlider(value=0.30, min=0.05, max=0.95, step=0.01,
    description="Design cutoff ωc/π:", readout_format=".2f", continuous_update=False,
    style={"description_width": "initial"}, layout=widgets.Layout(width="370px"))
custom_control = widgets.Text(value="0.25, 0.5, 0.25", description="b[k]:",
    placeholder="e.g. 0.25, 0.5, 0.25", continuous_update=False,
    style={"description_width": "initial"}, layout=widgets.Layout(width="370px"))

mag_mode = widgets.ToggleButtons(options=["dB", "Linear"], value="dB", description="Magnitude:",
    style={"description_width": "initial"})
phase_mode = widgets.ToggleButtons(options=["Unwrapped", "Wrapped"], value="Unwrapped",
    description="Phase:", style={"description_width": "initial"})
group_delay_control = widgets.Checkbox(value=False, description="Show group delay")
marker_control = widgets.FloatSlider(value=1/3, min=0, max=1, step=0.01,
    description="Reference ω/π:", readout_format=".2f", continuous_update=False,
    style={"description_width": "initial"}, layout=widgets.Layout(width="370px"))

param_widgets = {
    "N-point moving average": [n_taps_control],
    "Windowed-sinc low-pass": [n_taps_control, cutoff_control],
    "Windowed-sinc high-pass": [n_taps_control, cutoff_control],
    "Adjustable two-tap average": [mix_control],
    "Pure delay": [delay_control],
    "Symmetric 3-tap smoother": [symmetry_control],
    "Custom coefficients": [custom_control],
}

def show_active_parameters(change=None):
    active = param_widgets.get(filter_choice.value, [])
    for control in [n_taps_control, mix_control, symmetry_control,
                    delay_control, cutoff_control, custom_control]:
        control.layout.display = "" if control in active else "none"

filter_choice.observe(show_active_parameters, names="value")
show_active_parameters()


def render_fir(family, n_taps, mix, symmetry, delay, cutoff, custom,
               magnitude_mode, phase_display, group_delay, marker):
    try:
        b, note = make_fir(family, n_taps, mix, symmetry, delay, cutoff, custom)
    except ValueError as exc:
        print(f"Please correct the coefficients: {exc}")
        return

    w, h, amp, db, unwrapped = calculate_response(b)
    n = np.arange(len(b))
    w0 = float(marker) * np.pi
    h0 = np.sum(b * np.exp(-1j * w0 * n))
    symmetric = np.allclose(b, b[::-1])
    antisymmetric = np.allclose(b, -b[::-1])
    linear_phase = symmetric or antisymmetric

    n_rows = 3 if group_delay else 2
    fig = plt.figure(figsize=(11.5, 8.3 if group_delay else 6.5), layout="constrained")
    gs = fig.add_gridspec(n_rows, 2, width_ratios=[1.05, 1.9])
    ax_coeff = fig.add_subplot(gs[0, 0])
    ax_notes = fig.add_subplot(gs[1:, 0])
    ax_mag = fig.add_subplot(gs[0, 1])
    ax_phase = fig.add_subplot(gs[1, 1], sharex=ax_mag)

    ax_coeff.stem(n, b, basefmt="0.5", linefmt="C0-", markerfmt="C0o")
    ax_coeff.set(xlabel="Tap index k", ylabel="b[k]", title="Impulse response / coefficients")
    ax_coeff.grid(alpha=0.25)
    ax_coeff.set_xlim(-0.7, max(2.7, len(b)-0.3))
    # Most long filters do not benefit from every tap number on the axis.
    step = max(1, int(np.ceil(len(b) / 12)))
    ax_coeff.set_xticks(n[::step])

    values = db if magnitude_mode == "dB" else amp
    ax_mag.plot(w, values, color="C0", lw=2)
    ax_mag.axvline(w0, color="0.4", ls=":", lw=1.25)
    ax_mag.plot([w0], [20*np.log10(max(abs(h0), 1e-5)) if magnitude_mode == "dB" else abs(h0)],
                "o", ms=5, color="C3")
    ax_mag.set(title="Magnitude response", ylabel="Magnitude (dB)" if magnitude_mode == "dB" else "Magnitude (linear)")
    if magnitude_mode == "dB":
        ax_mag.set_ylim(-100, max(5, float(np.max(db))+3))
    else:
        ax_mag.set_ylim(bottom=0)
    ax_mag.grid(alpha=0.25)

    phase = np.angle(h) if phase_display == "Wrapped" else unwrapped.copy()
    # At response zeros the phase and group delay are undefined.
    phase = np.where(amp > max(np.max(amp) * 1e-5, 1e-8), phase, np.nan)
    ax_phase.plot(w, phase, color="C1", lw=1.8)
    ax_phase.axvline(w0, color="0.4", ls=":", lw=1.25)
    ax_phase.set(title="Phase response", ylabel="Phase (rad)")
    ax_phase.grid(alpha=0.25)

    last_ax = ax_phase
    if group_delay:
        ax_delay = fig.add_subplot(gs[2, 1], sharex=ax_mag)
        # Differentiate unwrapped phase; values near magnitude nulls are unreliable.
        delay_curve = -np.gradient(unwrapped, w)
        delay_curve[np.abs(h) < max(np.max(amp)*0.01, 1e-8)] = np.nan
        delay_curve[np.abs(delay_curve) > 2 * max(len(b), 2)] = np.nan
        ax_delay.plot(w, delay_curve, color="C2", lw=1.6)
        ax_delay.axvline(w0, color="0.4", ls=":", lw=1.25)
        ax_delay.set(title="Group delay (undefined at response zeros)",
                     ylabel="Samples")
        ax_delay.grid(alpha=0.25)
        last_ax = ax_delay

    for axis in [ax_mag, ax_phase]:
        if axis is not last_ax:
            axis.tick_params(labelbottom=False)
    last_ax.set(xlim=(0, np.pi), xlabel=r"Frequency $\omega$ (rad/sample)")
    last_ax.set_xticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
    last_ax.set_xticklabels(["0", r"$\pi/4$", r"$\pi/2$", r"$3\pi/4$", r"$\pi$"])

    ax_notes.axis("off")
    desc = [
        f"Number of taps: {len(b)}",
        f"H(0) = {np.sum(b):.3g}",
        f"|H(π)| = {abs(np.sum(b * (-1)**n)):.3g}",
        "",
        "LINEAR PHASE",
        "Yes (symmetric)" if symmetric else (
            "Yes (antisymmetric)" if antisymmetric else (
            "Yes (pure delay)" if family == "Pure delay" else "Not necessarily"
        )),
    ]
    if linear_phase:
        desc.append(f"Nominal delay: {(len(b)-1)/2:g} samples")
    elif family == "Pure delay":
        desc.append(f"Delay: {delay} samples")
    desc += ["", "REFERENCE MARKER", f"ω = {marker:.2f}π rad/sample",
             f"|H(ω)| = {abs(h0):.4f}",
             f"∠H(ω) = {np.angle(h0):+.3f} rad" if abs(h0) > 1e-8 else "∠H(ω) undefined (null)"]
    ax_notes.text(0.02, 0.97, "\n".join(desc), va="top", ha="left",
                  transform=ax_notes.transAxes, fontsize=10.5, linespacing=1.4,
                  family="monospace")
    fig.suptitle(family, fontsize=14)
    plt.show()
    plt.close(fig)
    print(note)


plot_output = widgets.interactive_output(render_fir, {
    "family": filter_choice,
    "n_taps": n_taps_control,
    "mix": mix_control,
    "symmetry": symmetry_control,
    "delay": delay_control,
    "cutoff": cutoff_control,
    "custom": custom_control,
    "magnitude_mode": mag_mode,
    "phase_display": phase_mode,
    "group_delay": group_delay_control,
    "marker": marker_control,
})

filter_panel = widgets.VBox([
    widgets.HTML("<b>Filter and coefficients</b>"), filter_choice, n_taps_control,
    mix_control, symmetry_control, delay_control, cutoff_control, custom_control,
])
view_panel = widgets.VBox([
    widgets.HTML("<b>Display options</b>"), mag_mode, phase_mode,
    group_delay_control, marker_control,
])
display(widgets.VBox([widgets.HBox([filter_panel, view_panel]), plot_output]))

## 2 · Filter and listen to a sound

Use the **same filter selected above** to process **white noise** or a **square wave**. Change either the FIR design or the sound controls; the waveform, power spectrum, and audio players update automatically.

- **Time plots:** compare input and output over the first 20 ms. The FIR filter can introduce a delay, so the waveforms need not align.
- **Spectra:** compare the input and output power spectra in dB. A frequency attenuated by the filter should generally have less power in the output spectrum. For noise, the spectrum is an estimate; the input and output share the same realization.
- **Listen:** play the original and filtered signals. The players use a **shared gain** and do *not* independently normalize the audio, so audible level differences reflect the filter gain.

The sound is sampled at **8 kHz**, so the Nyquist frequency (π rad/sample) corresponds to **4 kHz**: normalized frequency **ω/π = f / 4000 Hz**.

In [4]:
# The filter selector and its parameter widgets are shared with section 1.
# This second interactive section always uses the currently designed FIR filter.
SAMPLE_RATE = 8000

sound_choice = widgets.Dropdown(
    options=["White noise", "Square wave"], value="White noise",
    description="Sound:", style={"description_width": "initial"},
    layout=widgets.Layout(width="330px")
)
frequency_control = widgets.IntSlider(
    value=500, min=100, max=1600, step=50,
    description="Square frequency (Hz):", continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="360px")
)
duration_control = widgets.FloatSlider(
    value=1.0, min=0.5, max=2.0, step=0.25,
    description="Duration (s):", continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="360px")
)


def update_sound_controls(change=None):
    frequency_control.layout.display = "" if sound_choice.value == "Square wave" else "none"


sound_choice.observe(update_sound_controls, names="value")
update_sound_controls()


def render_sound(family, n_taps, mix, symmetry, delay, cutoff, custom,
                 sound_type, square_frequency, duration):
    try:
        b, _ = make_fir(family, n_taps, mix, symmetry, delay, cutoff, custom)
    except ValueError as exc:
        print(f"Please correct the FIR coefficients above: {exc}")
        return

    fs = SAMPLE_RATE
    n_samples = int(round(duration * fs))
    t = np.arange(n_samples) / fs
    if sound_type == "White noise":
        # Fixed seed: changing the filter does not change the input sound.
        x = np.random.default_rng(2026).uniform(-0.5, 0.5, size=n_samples)
    else:
        x = 0.5 * signal.square(2 * np.pi * square_frequency * t)

    # Causal FIR filtering: y[n] = sum_k b[k] x[n-k].
    y = signal.lfilter(b, [1.0], x)

    # Plot a short segment in the time domain (same vertical scale).
    segment_size = min(int(0.020 * fs), n_samples)
    segment_ms = 1000 * t[:segment_size]
    fig, (ax_time, ax_spectrum) = plt.subplots(
        1, 2, figsize=(12, 3.8), layout="constrained"
    )
    ax_time.plot(segment_ms, x[:segment_size], label="Input", lw=1.4)
    ax_time.plot(segment_ms, y[:segment_size], label="Filtered output", lw=1.3, alpha=0.85)
    ax_time.set(title="Waveforms (first 20 ms)", xlabel="Time (ms)", ylabel="Amplitude")
    ax_time.grid(alpha=0.25)
    ax_time.legend(loc="upper right")

    # Use the same PSD settings for input and output.
    nperseg = min(2048, n_samples)
    f, pxx = signal.welch(x, fs=fs, nperseg=nperseg, nfft=4096)
    _, pyy = signal.welch(y, fs=fs, nperseg=nperseg, nfft=4096)
    floor = 1e-14  # avoids log10(0) for spectral nulls
    ax_spectrum.plot(f, 10 * np.log10(np.maximum(pxx, floor)), label="Input", lw=1.3)
    ax_spectrum.plot(f, 10 * np.log10(np.maximum(pyy, floor)), label="Filtered output", lw=1.3)
    ax_spectrum.set(title="Power spectra (Welch estimate)",
                    xlabel="Frequency (Hz)", ylabel="PSD (dB/Hz)", xlim=(0, fs / 2))
    ax_spectrum.grid(alpha=0.25)
    ax_spectrum.legend(loc="upper right")
    plt.show()
    plt.close(fig)

    # IPython Audio normally normalizes each clip independently. That would
    # hide filter attenuation. Keep the relative gain identical in both players.
    peak = max(float(np.max(np.abs(x))), float(np.max(np.abs(y))), 1e-12)
    common_gain = min(1.0, 0.95 / peak)
    x_play = x * common_gain
    y_play = y * common_gain

    display(HTML("<b>Original sound</b>"))
    display(Audio(x_play, rate=fs, normalize=False))
    display(HTML("<b>Filtered sound</b>"))
    display(Audio(y_play, rate=fs, normalize=False))
    print(f"Current FIR: {family} | {len(b)} taps | input/output use the same playback gain")


sound_output = widgets.interactive_output(render_sound, {
    "family": filter_choice,
    "n_taps": n_taps_control,
    "mix": mix_control,
    "symmetry": symmetry_control,
    "delay": delay_control,
    "cutoff": cutoff_control,
    "custom": custom_control,
    "sound_type": sound_choice,
    "square_frequency": frequency_control,
    "duration": duration_control,
})

display(widgets.VBox([
    widgets.HBox([sound_choice, frequency_control, duration_control]),
    sound_output,
]))